# 06 — Imágenes como secuencias: RNN, LSTM, GRU y atención

**Pregunta:** ¿puede una red recurrente clasificar prendas leyendo la foto **columna a columna**, como si fuera una frase?

La foto RGB64 se lee de izquierda a derecha: **64 pasos**, cada uno con una columna de **192 valores** (64 píxeles × 3 colores). La red guarda un *estado* que resume lo leído hasta ese paso.

Hay tres fases, cada una guardada por separado:
1. **Celdas:** RNN simple, LSTM y GRU con el mismo estado de 128.
2. **Atención:** la misma LSTM, pero la clase sale de una media ponderada de todos sus estados.
3. **Orden de lectura (H6):** la misma LSTM leyendo filas en lugar de columnas.

> **Cómo leer este cuaderno.** Cada sección explica qué se hace y cómo leer el resultado. Las figuras están en inglés; las tablas y explicaciones, en castellano. Test no se abre en ningún momento.

| Palabra | Qué significa aquí |
|---|---|
| **Secuencia** | La foto convertida en 64 pasos ordenados (columnas) |
| **Estado** | Vector de 128 números que resume lo leído hasta ese paso |
| **RNN simple** | Combina el estado anterior con el paso nuevo; con secuencias largas tiende a «olvidar» el principio |
| **LSTM** | Añade una memoria y puertas que deciden qué guardar, qué olvidar y qué mostrar |
| **GRU** | Versión más sencilla de la LSTM, con dos puertas en lugar de tres |
| **Último estado** | Usar solo el estado final para decidir la clase |
| **Atención** | Dar un peso a cada paso (suman 1) y decidir con la media ponderada de todos los estados |

In [ ]:
from pathlib import Path
import sys, json, inspect, time
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/data/local_dataset.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.models.sequence import SequenceClassifier, multiply_adds
from src.models.cnn import multiply_adds as cnn_multiply_adds
from src.models.dense import DenseClassifier, parameter_count
from src.training.dense_experiment import load_context, prepare_arrays, normalized_tensors, grouped_intervals
from src.training.saved_experiments import load_saved_experiment
from src.training import cnn_experiment
from src.training.sequence_experiment import load_sequence_context, load_saved_phase, run_phase, model_from_config
from src.visualization.style import apply_style, format_number as number, format_percent as percent
from src.visualization.tables import display_es
from src.visualization.dense import show_compact, show_code, run_with_progress
from src.visualization import cnn as common, sequence as figures
apply_style()

# Banderas: Run All solo carga resultados. Entrenar exige activarlas expresamente.
ENTRENAR_CELDAS = False      # fase cells: RNN, LSTM y GRU (9 entrenamientos)
ENTRENAR_ATENCION = False    # fase attention: solo después de terminar cells
ENTRENAR_FILAS = False       # fase rows: solo después de terminar cells

context = load_sequence_context(ROOT)
config = context['config']
names, rows = context['names'], context['rows']
FOLDER = figures.FOLDER
eda = ROOT/'results/metrics/03_eda/dataset_provenance.json'
if not eda.exists() or json.loads(eda.read_text())['manifest_sha256'] != context['manifest_sha256']:
    raise ValueError('Ejecuta primero 03 con la publicación vigente de 02.')
# Referencias sin entrenar: la Dense de 04 (obligatoria) y la CNN de 05 (si ya está guardada).
dense_summary, DENSE = load_saved_experiment(load_context(ROOT))
cnn_saved = cnn_experiment.load_saved_phase(cnn_experiment.load_cnn_context(ROOT), 'h1')
saved_cells = None if ENTRENAR_CELDAS else load_saved_phase(context, 'cells')
if saved_cells is None and not ENTRENAR_CELDAS:
    raise FileNotFoundError('No hay resultados guardados de la fase cells para esta publicación. '
                            'Para entrenar, poner ENTRENAR_CELDAS = True: no se entrena por faltar una caché.')
torch.set_num_threads(config['training']['threads'])
torch.use_deterministic_algorithms(True)
display_es(pd.DataFrame({
    'Decisión': ['PyTorch', 'Dispositivo', 'Hilos CPU', 'Semillas', 'Batch', 'Máximo épocas', 'Paciencia',
                 'Learning rate', 'Estado oculto', 'Pesos de clase', 'Aumentos'],
    'Valor': [torch.__version__, 'CPU', config['training']['threads'], '42, 123 y 2026', 32, 30, 5, '0,001',
              config['architecture']['hidden_size'], 'No', 'No'],
}))
if cnn_saved is None:
    display(Markdown('**CNN de 05 todavía sin resultados guardados:** se compara solo con la Dense de 04.'))

## 1. Mismos datos que la Dense y la CNN

Usamos la misma publicación de 02 y **exactamente las mismas fotos** de train y validation que 04 y 05.

**Cómo leerlo:** si una comprobación falla, el cuaderno se detiene.

In [ ]:
display(Markdown(f"**Publicación:** `{context['run'].name}`. Manifiesto verificado por SHA-256."))
display_es(pd.DataFrame({'Partición': ['train', 'validation', 'test'],
                         'Imágenes': [context['split_counts'][s] for s in ['train', 'validation', 'test']],
                         'Uso aquí': ['Aprender pesos y normalización', 'Parada, checkpoint y análisis',
                                      'Reservado; no se abren fotografías']}))
dense_predictions = pd.read_csv(DENSE/'baseline_seed42_predictions.csv.gz', dtype={'article_id': 'string'})
assert dense_predictions.article_id.tolist() == rows['validation'].article_id.tolist()
missing_validation = sorted(set(names) - set(rows['validation'].label))
if missing_validation:
    display(Markdown(f"**Clases sin fotos en validation:** {', '.join(missing_validation)}. Se aprenden en train, "
                     f"pero no se pueden evaluar. Macro-F1 promedia las {len(names)} clases, como en 03–05, así que cuentan con F1 = 0."))
print('Controles superados: misma publicación y mismas fotos de validation que la Dense de 04.')

## 2. De foto a secuencia

Cada columna de la foto se convierte en un paso: sus 64 píxeles por 3 colores dan 192 valores. Los píxeles se normalizan con train **antes** de construir la secuencia, igual que en 04 y 05.

**Cómo leer la figura:** a la izquierda, la foto con líneas cada 8 columnas; a la derecha, la secuencia que recibe la red: cada columna de la imagen derecha es un paso.

In [ ]:
prepared_at = time.perf_counter()
raw, normalization = prepare_arrays(context)
tensors = normalized_tensors(raw, normalization)
display(Markdown(f"Preparación/verificación: **{number(time.perf_counter() - prepared_at, 1)} s** (misma caché RGB64 que 04)."))
show_compact(figures.sequence_figure(context, raw, tensors), width=800)
display_es(pd.DataFrame({'Lectura': ['Columnas (principal)', 'Filas (fase 3)'],
                         'Pasos': [64, 64], 'Valores por paso': [192, 192],
                         'Orden': ['De izquierda a derecha', 'De arriba abajo']}))

## 3. Las tres celdas

Las tres tienen **el mismo estado de 128**: la misma «memoria». Cambia cómo la actualizan:
- **RNN simple:** una sola combinación del estado y del paso.
- **GRU:** dos puertas (actualizar y reiniciar).
- **LSTM:** tres puertas y una memoria aparte.

Más puertas significa más parámetros y más operaciones.

**Cómo leer la tabla:** las redes recurrentes tienen **muchos menos parámetros** que la Dense y la CNN. Por eso la comparación con ellas es de referencia, no un contraste igualado como H1.

In [ ]:
classes = len(names)
example = SequenceClassifier('lstm', readout='attention', classes=classes)
show_code(inspect.getsource(SequenceClassifier), 'Ver el código del modelo secuencial')
sizes = []
for label, variant in [('RNN simple', 'rnn'), ('LSTM', 'lstm'), ('GRU', 'gru')]:
    m = SequenceClassifier(variant, classes=classes)
    sizes.append({'Modelo': label, 'Parámetros': parameter_count(m), 'Multiplicaciones-sumas por imagen': multiply_adds(m)})
sizes.append({'Modelo': 'LSTM + atención', 'Parámetros': parameter_count(example), 'Multiplicaciones-sumas por imagen': multiply_adds(example)})
dense_model = DenseClassifier(classes=classes)
cnn_model = cnn_experiment.model_from_config(cnn_experiment.load_cnn_context(ROOT)['config'], {'batchnorm': False})
sizes += [{'Modelo': 'Dense (04, referencia)', 'Parámetros': parameter_count(dense_model),
           'Multiplicaciones-sumas por imagen': cnn_multiply_adds(dense_model)},
          {'Modelo': 'CNN (05, referencia)', 'Parámetros': parameter_count(cnn_model),
           'Multiplicaciones-sumas por imagen': cnn_multiply_adds(cnn_model)}]
display_es(pd.DataFrame(sizes))

## 4. Entrenar o cargar las celdas

En el modo normal el cuaderno **carga** las nueve ejecuciones guardadas (tres celdas × tres semillas). Para entrenar se pone `ENTRENAR_CELDAS = True`.

Antes de entrenar se mide **una época real de cada celda** (que se descarta). Si la estimación de las nueve ejecuciones supera el presupuesto de la fase (5 horas), no empieza y no se quita ninguna foto.

**Cómo leer las tablas:** *época elegida* es la de mejor macro-F1 en validation; *épocas realizadas* incluye la parada temprana.

In [ ]:
if ENTRENAR_CELDAS:
    cells_summary, CELLS = run_with_progress(lambda: run_phase(context, tensors, normalization, 'cells'),
                                             ROOT/'tmp/06_training_cells.log')
else:
    cells_summary, CELLS = saved_cells
records = pd.DataFrame(cells_summary['runs'])
assert cells_summary['test_images_opened'] == 0 and cells_summary['test_predictions_generated'] == 0
assert set(records.variant) == {'rnn', 'lstm', 'gru'} and len(records) == 9
feasibility = cells_summary['cost_feasibility']
display_es(pd.DataFrame({'Medición': ['Época piloto media (s)', 'Estimación conservadora de la fase (min)', 'Presupuesto (min)'],
                         'Valor': [feasibility['pilot_epoch_seconds'], feasibility['conservative_upper_seconds']/60,
                                   feasibility['budget_seconds']/60]}))
display(Markdown(f"**Fase cells completada:** `{cells_summary['experiment_id']}`. Resultados en `results/metrics/06_sequence/runs/{CELLS.name}`."))
display_es(records[['variant', 'seed', 'train_macro_f1', 'validation_macro_f1', 'validation_accuracy']])
display_es(records[['variant', 'seed', 'selected_epoch', 'epochs_run', 'train_seconds']])

## 5. Curvas de aprendizaje

Una figura por celda. **Morado: train. Naranja: validation.** Cada estilo de línea es una semilla.

**Cómo leerlas:** si la RNN simple se queda plana mientras LSTM y GRU mejoran, es el síntoma típico de que no consigue recordar 64 pasos.

In [ ]:
for variant, title in [('rnn', 'How the simple RNN learns'), ('lstm', 'How the LSTM learns'), ('gru', 'How the GRU learns')]:
    show_compact(common.curves_figure(context, CELLS, variant, title, folder=FOLDER), width=820)

## 6. Comparar las celdas

Comparamos **semilla a semilla** con la Dense de 04 (y con la CNN de 05 si ya está entrenada). El intervalo de confianza remuestrea **productos** de validation con las predicciones de las dos redes a la vez.

**Cómo leerlo:** cada marcador es una semilla y la raya, la media. Si el intervalo de la diferencia con la Dense no incluye el 0, la diferencia es consistente en esta muestra de validation.

In [ ]:
labels = {'rnn': 'RNN', 'lstm': 'LSTM', 'gru': 'GRU'}
table = pd.DataFrame({'Dense (04)': pd.DataFrame(dense_summary['runs']).query("variant == 'baseline'").set_index('seed').validation_macro_f1})
if cnn_saved is not None:
    table['CNN (05)'] = pd.DataFrame(cnn_saved[0]['runs']).set_index('seed').validation_macro_f1
for variant, label in labels.items():
    table[label] = records.query('variant == @variant').set_index('seed').validation_macro_f1
table.index.name = 'Semilla'
display_es(table)
show_compact(figures.models_figure(context, table, '03_cells_comparison', 'Reading columns: RNN, LSTM and GRU',
             'Source: validation macro-F1 of the selected checkpoint; same photos and training budget. Recurrent '
             'models have a 128 state and far fewer parameters than Dense and CNN.'), width=780)
y = rows['validation'].label_id.astype(int).to_numpy()
groups = rows['validation'].group_id
contrast = []
for variant, label in labels.items():
    for seed in config['training']['seeds']:
        pred = pd.read_csv(CELLS/f'{variant}_seed{seed}_predictions.csv.gz').predicted_label_id.to_numpy()
        dense_pred = pd.read_csv(DENSE/f'baseline_seed{seed}_predictions.csv.gz').predicted_label_id.to_numpy()
        ci = grouped_intervals(y, pred, groups, len(names), draws=config['bootstrap']['draws'],
                               seed=config['bootstrap']['seed'], other=dense_pred)
        contrast.append({'Modelo': label, 'Semilla': seed, 'Diferencia con Dense': table.loc[seed, label] - table.loc[seed, 'Dense (04)'],
                         'IC95 inferior': ci[0][1], 'IC95 superior': ci[1][1]})
contrast = pd.DataFrame(contrast)
display_es(contrast)
table.to_csv(CELLS/'analysis_cells_by_seed.csv')
contrast.to_csv(CELLS/'analysis_cells_vs_dense.csv', index=False)
means = table.mean()
display(Markdown('Macro-F1 medio: ' + ', '.join(f"**{k}** {number(v, 3)}" for k, v in means.items()) + '.'))

## 7. Dónde acierta y dónde falla la LSTM

Usamos la **LSTM, semilla 42**, fijada en la configuración antes de entrenar: no se elige la mejor.

**Cómo leerlo:** las barras comparan el F1 de cada clase con la Dense; la matriz muestra qué clases se confunden (filas = clase real); la galería enseña los errores con más confianza, elegidos automáticamente.

In [ ]:
from sklearn.metrics import confusion_matrix
lstm_predictions = pd.read_csv(CELLS/'lstm_seed42_predictions.csv.gz', dtype={'article_id': 'string', 'group_id': 'string'})
lstm_report = json.loads((CELLS/'lstm_seed42_per_class.json').read_text(encoding='utf-8'))
dense_report = json.loads((DENSE/'baseline_seed42_per_class.json').read_text(encoding='utf-8'))
per_class = pd.DataFrame({'F1 Dense': [dense_report[n]['f1-score'] for n in names],
                          'F1 LSTM': [lstm_report[n]['f1-score'] for n in names],
                          'Imágenes': [lstm_report[n]['support'] for n in names]}, index=pd.Index(names, name='Clase'))
per_class['Diferencia'] = per_class['F1 LSTM'] - per_class['F1 Dense']
display_es(per_class.sort_values('Diferencia'))
show_compact(common.per_class_figure(context, dense_report, lstm_report, ('Dense (04, seed 42)', 'LSTM (seed 42)'),
             'Where the LSTM gains or loses against the Dense network', folder=FOLDER), width=900)
cm = confusion_matrix(y, lstm_predictions.predicted_label_id, labels=np.arange(len(names)))
show_compact(common.confusion_figure(context, cm, 'LSTM', '04_confusion', folder=FOLDER), width=900)
show_compact(common.errors_figure(context, lstm_predictions, model_name='LSTM', folder=FOLDER), width=720)

## 8. Fase 2 · Atención

La misma LSTM, mismas semillas y datos; **solo cambia cómo se resume la secuencia**: en lugar del último estado, una media ponderada de los 64 estados. Los pesos los aprende una pequeña red y siempre suman 1. Se entrena con `ENTRENAR_ATENCION = True`, después de la fase de celdas.

**Cómo leerlo:** la primera figura compara semilla a semilla. En el mapa, cada barra es el peso de una columna; la línea discontinua es el peso uniforme (1/64). Los pesos dicen **dónde mira** el resumen, no **por qué** decide una clase.

In [ ]:
saved_attention = None if ENTRENAR_ATENCION else load_saved_phase(context, 'attention')
if ENTRENAR_ATENCION:
    saved_attention = run_with_progress(lambda: run_phase(context, tensors, normalization, 'attention'),
                                        ROOT/'tmp/06_training_attention.log')
attention_records = None
if saved_attention is None:
    display(Markdown('**Fase de atención pendiente.** Activar `ENTRENAR_ATENCION = True` cuando se decida entrenarla.'))
else:
    attention_summary, ATTENTION = saved_attention
    attention_records = pd.DataFrame(attention_summary['runs'])
    assert attention_summary['test_images_opened'] == 0
    show_compact(common.curves_figure(context, ATTENTION, 'lstm_attention', 'How the LSTM with attention learns',
                 folder=FOLDER), width=820)
    attention_pairs = pd.DataFrame({'LSTM · último estado': table['LSTM'],
                                    'LSTM · atención': attention_records.set_index('seed').validation_macro_f1})
    attention_pairs.index.name = 'Semilla'
    attention_pairs['Diferencia'] = attention_pairs['LSTM · atención'] - attention_pairs['LSTM · último estado']
    display_es(attention_pairs)
    show_compact(common.paired_figure(context, attention_pairs['LSTM · último estado'], attention_pairs['LSTM · atención'],
                 ['Last state', 'Attention'], '07_attention_pairs', 'A single change: attention over all states',
                 'Source: validation macro-F1 of the selected checkpoint; same LSTM, seeds, data and budget.',
                 folder=FOLDER), width=760)
    attention_pairs.to_csv(ATTENTION/'analysis_attention_pairs.csv')
    reference = attention_records.query('seed == 42').iloc[0]
    model, checkpoint = common.checkpoint_model(context, reference, model_from_config,
                                                config['phases']['attention']['variants'][0])
    if model is None:
        display(Markdown(f"**Pesos no disponibles en este equipo:** `{checkpoint.name}`. El mapa de atención queda pendiente."))
    else:
        show_compact(figures.attention_figure(context, model, tensors), width=850)

## 9. Fase 3 · Orden de lectura (H6)

La misma LSTM leyendo **filas** (de arriba abajo) en lugar de columnas. Mismo número de pasos y de valores por paso; solo cambia qué vecindario forma cada paso. Se entrena con `ENTRENAR_FILAS = True`, después de la fase de celdas.

**Cómo leerlo:** si las filas ganan o pierden de forma consistente, el resultado de una red recurrente depende de cómo se convierte la imagen en secuencia, no solo de la celda.

In [ ]:
saved_rows = None if ENTRENAR_FILAS else load_saved_phase(context, 'rows')
if ENTRENAR_FILAS:
    saved_rows = run_with_progress(lambda: run_phase(context, tensors, normalization, 'rows'), ROOT/'tmp/06_training_rows.log')
rows_records = None
if saved_rows is None:
    display(Markdown('**Fase de filas pendiente.** Activar `ENTRENAR_FILAS = True` cuando se decida entrenarla.'))
else:
    rows_summary, ROWS = saved_rows
    rows_records = pd.DataFrame(rows_summary['runs'])
    assert rows_summary['test_images_opened'] == 0
    order_pairs = pd.DataFrame({'LSTM · columnas': table['LSTM'],
                                'LSTM · filas': rows_records.set_index('seed').validation_macro_f1})
    order_pairs.index.name = 'Semilla'
    order_pairs['Diferencia'] = order_pairs['LSTM · filas'] - order_pairs['LSTM · columnas']
    display_es(order_pairs)
    show_compact(common.paired_figure(context, order_pairs['LSTM · columnas'], order_pairs['LSTM · filas'],
                 ['Columns', 'Rows'], '08_reading_order_pairs', 'H6: does the reading order matter?',
                 'Source: validation macro-F1 of the selected checkpoint; same LSTM, seeds, data and budget.',
                 folder=FOLDER), width=760)
    order_pairs.to_csv(ROWS/'analysis_reading_order_pairs.csv')

## 10. Coste y procedencia

**Cómo leerlo:** el tiempo de entrenamiento incluye la evaluación de cada época; no incluye preparar las fotos. La memoria es la de todo el proceso de Python.

In [ ]:
all_records = pd.concat([r for r in [records, attention_records, rows_records] if r is not None])
display_es(common.cost_frame(all_records))
display(Markdown(f"**Entrenamiento total guardado:** {number(all_records.train_seconds.sum() / 60, 1)} minutos. "
                 f"Hardware: `{cells_summary['hardware']['processor']}`, CPU con {cells_summary['hardware']['threads_used']} hilos."))
display_es(pd.DataFrame({'Elemento': ['Dataset', 'Configuración', 'Código del modelo', 'PyTorch', 'Referencia Dense'],
                         'Referencia': [cells_summary['dataset_run'], cells_summary['identity']['sources']['config/sequence_v1.json'],
                                        cells_summary['identity']['sources']['src/models/sequence.py'],
                                        cells_summary['identity']['versions']['torch'], dense_summary['experiment_id']]}))

## 11. Conclusiones

Las cifras se calculan a partir de los resultados anteriores; no se rellenan con resultados supuestos.

In [ ]:
lines = [f"- **Celdas (macro-F1 medio):** RNN {number(means['RNN'], 3)}, LSTM {number(means['LSTM'], 3)}, "
         f"GRU {number(means['GRU'], 3)}; Dense de referencia {number(means['Dense (04)'], 3)}."]
if 'CNN (05)' in means:
    lines.append(f"- **CNN de referencia:** {number(means['CNN (05)'], 3)}.")
for label in ['RNN', 'LSTM', 'GRU']:
    d = contrast.query('Modelo == @label')['Diferencia con Dense']
    lines.append(f"- **{label} − Dense:** diferencia media {number(d.mean(), 3)}; mejor que la Dense en {int((d > 0).sum())}/3 semillas.")
lines.append('- **Atención:** ' + (f"diferencia media {number(attention_pairs['Diferencia'].mean(), 3)}; mejora en "
             f"{int((attention_pairs['Diferencia'] > 0).sum())}/3 semillas." if attention_records is not None else 'fase pendiente.'))
lines.append('- **Orden de lectura:** ' + (f"filas − columnas {number(order_pairs['Diferencia'].mean(), 3)} de media."
             if rows_records is not None else 'fase pendiente.'))
display(Markdown('\n'.join(lines)))

### Límites

- Son resultados de **validation**, que también se usa para elegir la época: no sustituyen la evaluación final en test.
- Las redes recurrentes tienen el mismo estado (128) pero **muchos menos parámetros** que la Dense y la CNN: la comparación con ellas es de referencia.
- Una sola capa, unidireccional. Las variantes bidireccionales, apiladas o CNN + RNN quedan como extensiones; los parches 8×8 no se ejecutan porque cambian el vecindario.
- Los pesos de atención muestran dónde mira el resumen, no explican la predicción.
- La evaluación describe las prendas seleccionadas tras los controles, no todo el catálogo.

**Material de clase:** `PRDL_MLLB_RecurrentNetworks.pptx`, `MLLB_Embeddings_RNN_SentimentAnalysis.ipynb`, `TO_DO_MLLB_Embeddings_TwoLayers_RNN_SentimentAnalysis.ipynb`, diapositiva «Image: RNN (LSTM, GRU…)». [Atención y recurrencia](https://arxiv.org/abs/1706.03762).